# 05 — GATE RUN: ImageNet-1k, banyak skenario sekaligus

**Ini RUN GERBANG, bukan run paper.** Ia bisa gagal dan menghentikan pengembangan metode.
Kriterianya ditetapkan di `reports/prereg_imagenet_gate.md`, **ditulis sebelum data diunduh**.
Baca dokumen itu dulu — notebook ini tidak boleh dipakai untuk memilih kriteria.

## Mengapa ImageNet, dan mengapa ini penyimpangan

Urutan pre-registered menyatakan dataset berikutnya hanya dijalankan jika Phase 1 lulus di
Pl@ntNet. **Gate B/C tidak lulus.** Jadi ini penyimpangan, dicatat sebagai keputusan eksplisit.

Alasannya daya uji, dan itu terukur tanpa melihat hasil Pl@ntNet:

| dump skor | sampel x kelas | per kelas | kelas dgn delta_y (n_cal=25) |
|---|---|---|---|
| LTC plantnet cal | 21.783 x 1.081 | median **3** | **152** -> 38 per stratum |
| CCC imagenet | 115.301 x 1.000 | **115** | **1.000**, tanpa stratifikasi |

Uji permutasi tingkat-kelas, terkalibrasi: pada 38 kelas sinyal lemah memberi p=0,066 (tidak
terdeteksi); pada 1.000 kelas sinyal lemah yang sama memberi p=0,0066.

ImageNet juga **berimbang**, jadi confound kualitas-deskriptor <-> prevalensi yang memaksa
Amandemen 5 tidak ada di sini — stratifikasi tidak diperlukan, bukan dihindari.

## Yang dibundel dalam satu run

gate A - gate B (bootstrap tingkat-KELAS) - gate C (permutasi tingkat-KELAS + Holm) -
multi-alpha {0,01; 0,05; 0,10} - dua n_cal - Sec 6.4 - reproduksi Clustered CP - perbandingan
berdampingan dengan Pl@ntNet.

## Yang HARUS diingat saat membaca hasilnya

phi(y) di sini adalah geometri **ruang OUTPUT**, dihitung dari matriks skor saja — bukan phi(y)
embedding yang dipakai di Pl@ntNet. Hasil di sini **tidak otomatis berpindah** ke deskriptor
embedding. Yang ia jawab: apakah geometri tingkat-kelas memprediksi delta_y begitu jumlah
kelasnya memadai.


## 1. Config — `# === EDIT ME ===`


In [ ]:
# === EDIT ME ===========================================================
REPO_URL   = ''
REPO_DIR   = 'foundation-cp'
DRIVE_ROOT = '/content/drive/MyDrive/pcc'

# SUMBER DUMP. LTC sudah punya ID gdown konkret (dari notebook 00) -> nol langkah
# manual. CCC belum: ID-nya harus dibaca dari download_data.sh mereka. Jadi survei
# LTC dulu; pindah ke CCC hanya kalau tidak ada dump LTC yang memenuhi premis.
# TIDAK ADA saklar sumber. Versi sebelumnya punya SOURCE yang harus diedit manual,
# dan default-nya ('ltc') sudah diketahui GAGAL premis — jadi setiap run default
# berakhir dengan assert. Sekarang SEMUA sumber disurvei dalam satu jalan dan yang
# terbaik dipakai. Tabel perbandingannya sendiri adalah temuan yang dicari.
CCC_DATASETS = ('imagenet',)   # tambah 'inaturalist' (iNat-2021, 633 kelas) bila perlu
GID_SCORES_LTC = {'plantnet': '1k_PPQV3VJT44hz02CcnbqPstjQo70vGr',
                  'inaturalist': '1W8R8Jj2bhS2PbR-3X9vEw-WkanbOk6mq'}
# ID CCC dari download_data.sh mereka. Ditanam supaya TIDAK ada langkah manual:
# versi sebelumnya hanya mencetak skripnya dan menyuruh menjalankan gdown sendiri,
# yang membingungkan dan tidak perlu begitu ID-nya diketahui.
GID_SCORES_CCC = {'imagenet':   '1AQjUn3m010N_i6-sfD690W7mZq2RTwJz',   # 4,62 GB
                  'inaturalist':'1BUlQZhS_5x2LJpyxCGI1IkmRrkvmRD88',   # 6,72 GB (iNat-2021, 633 kelas)
                  'places365':  '119k7PE1l72fg5Rpez5brIOn28BwClqv2',   # 0,54 GB (365 kelas)
                  'cifar-100':  '1yXD9XqBxEJnJxcfnnduNK6nHHUU3iX_6'}   # 0,01 GB (100 kelas)
# Catatan daya uji: premis butuh >=500 kelas layak, jadi places365 (365 kelas) dan
# cifar-100 (100 kelas) TIDAK BISA memenuhinya secara konstruksi, berapa pun
# sampel per kelasnya. Yang mungkin: imagenet (1.000) dan iNat-2021 (633).
LTC_DATASETS = ('inaturalist', 'plantnet')   # rilis memuat varian -trunc juga
LOSS_VARIANT = 'cross_entropy'  # 'cross_entropy' | 'focal'. LTC mengirim SEMBILAN
                                # berkas dengan NAMA IDENTIK di subdirektori berbeda;
                                # tercampur = skor dari model lain, akurasi mirip,
                                # kalibrasi beda total. Notebook 00 kena isu yang sama.
# places365 (365 kelas) dan cifar-100 (100 kelas) TIDAK BISA memenuhi premis >=500
# kelas secara konstruksi, berapa pun sampel per kelasnya — jadi tidak diunduh.
N_CLASSES_EXPECTED = None      # None = jangan dipaksakan; dibaca dari dump

# --- PRIMER, ditetapkan di prereg_imagenet_gate.md. JANGAN diubah setelah melihat hasil.
ALPHA_PRIMARY = 0.10
N_CAL_PRIMARY = 25
N_BOOT_CLASS  = 400           # bootstrap tingkat-kelas untuk gate B
N_PERM_CLASS  = 1000          # permutasi tingkat-kelas untuk gate C (p_min = 1/1001)
STABLE_THRESHOLD = 0.90

# --- SEKUNDER
ALPHAS_SECONDARY = (0.01, 0.05)
N_CAL_SECONDARY  = 50
N_SPLITS_BC = 100
N_SPLITS_A  = 100
RUN_CLUSTERED_CP = True       # reproduksi baseline pada skor yang sama

FRAC_DESC, FRAC_CAL = 0.40, 0.30   # sisanya EVAL

# ANGGARAN BARIS. Dump ImageNet CCC nyata adalah (1.153.051 x 1.000) float32 = 4,61 GB
# -- sepuluh kali lebih besar dari yang tercatat di release_audit.md. Memuatnya penuh
# lalu membuat salinan turunan (thr_lac, entropi, np.partition) melewati RAM Colab.
#
# Subsampling di sini BUKAN perubahan kriteria: premis butuh >=84 sampel/kelas dan
# anggaran ini menyisakan ~230/kelas. Ia diambil sebagai FRAKSI per kelas, bukan cap
# tetap, karena cap tetap membuat semua hitungan kelas SAMA -> log_prevalence konstan
# -> ablasi prevalensi jadi hampa. Fraksi mempertahankan struktur prevalensinya.
MAX_ROWS = 250_000            # None = pakai seluruh dump
SEED = 42
# =======================================================================
print(f'PRIMER: alpha={ALPHA_PRIMARY} n_cal={N_CAL_PRIMARY} '
      f'n_boot={N_BOOT_CLASS} n_perm={N_PERM_CLASS}')


## 2. Mount + repo + env


In [ ]:
import os, subprocess
from google.colab import drive
drive.mount('/content/drive')
if REPO_URL and not os.path.isdir(REPO_DIR):
    subprocess.run(['git','clone',REPO_URL,REPO_DIR], check=True)
if os.path.isdir(REPO_DIR):
    os.chdir(REPO_DIR if os.path.isabs(REPO_DIR) else '/content/'+REPO_DIR)
os.environ['PYTHONPATH'] = os.getcwd() + os.pathsep + os.environ.get('PYTHONPATH','')
subprocess.run(['pip','install','-q','-r','requirements.txt'], check=False)
subprocess.run(['pip','install','-q','gdown'], check=False)
from pcc.utils.seed import set_seed; from pcc.utils.io import environment_stamp
set_seed(SEED)
print('env:', environment_stamp()['packages'])


## 3. Siapkan SEMUA dump — TANPA citra, TANPA GPU

Dump LTC dipakai lokasi yang sama dengan notebook 00 (`released_scores/<dataset>`), jadi kalau
sudah ada tidak diunduh ulang. Dump CCC diunduh otomatis dengan ID yang sudah ditanam.

Keduanya disurvei berdampingan di sel 4. Itu bukan pemborosan: **perbandingan ekor-panjang
versus berimbang adalah temuannya**, dan menurunkannya dari satu tabel lebih kuat daripada dari
dua run terpisah.


In [ ]:
import glob, zipfile, numpy as np

def ltc_dir(ds):
    return f'{DRIVE_ROOT}/released_scores/{ds}'

def ccc_dir(ds):
    return f'{DRIVE_ROOT}/scores_ccc/{ds}'

for ds in LTC_DATASETS:
    d = ltc_dir(ds); os.makedirs(d, exist_ok=True)
    if glob.glob(f'{d}/**/*_softmax.npy', recursive=True):
        print(f'ltc/{ds}: sudah ada, dilewati')
        continue
    print(f'ltc/{ds}: mengunduh...')
    z = f'{d}/{ds}.zip'
    r = subprocess.run(['gdown', GID_SCORES_LTC[ds], '-O', z], capture_output=True, text=True)
    if r.returncode:
        print('  gdown gagal:', r.stderr.strip()[:300])
    else:
        subprocess.run(['unzip','-o','-q',z,'-d',d], check=False)

def inventory(d, label):
    files = [p for p in sorted(glob.glob(f'{d}/**/*', recursive=True)) if os.path.isfile(p)]
    print(f'  isi {label}: {len(files)} berkas')
    for p in files[:25]:
        print(f'    {os.path.relpath(p, d):56s} {os.path.getsize(p)/1e6:9.2f} MB')
    return files

def looks_like_html(p):
    # Kegagalan kuota Google Drive menulis halaman HTML DAN mengembalikan kode 0.
    # Inilah sebabnya returncode tidak boleh dipercaya sebagai bukti unduhan berhasil.
    try:
        with open(p, 'rb') as fh:
            head = fh.read(400)
    except OSError:
        return False, b''
    low = head.lower()
    return (b'<html' in low or b'<!doctype html' in low), head

for ds in CCC_DATASETS:
    d = ccc_dir(ds); os.makedirs(d, exist_ok=True)
    mat = f'/content/ccc_npy/{ds}'
    # Tiga keadaan, dan versi sebelumnya hanya mengenali yang pertama:
    #   (a) .npy sudah dimaterialkan di /content -> tidak ada kerja
    #   (b) .npz ada di Drive tapi .npy hilang (sesi baru; /content ephemeral)
    #       -> ekstrak ulang, JANGAN unduh 4,6 GB lagi
    #   (c) tidak ada apa pun -> unduh
    # Pemeriksaan lama hanya mencari .npy DI DRIVE, yang tidak pernah ada karena
    # materialisasinya ke /content. Jadi setiap sesi baru mengunduh ulang 4,6 GB.
    if glob.glob(f'{mat}/*.npy') or glob.glob(f'{d}/**/*.npy', recursive=True):
        print(f'ccc/{ds}: .npy sudah ada, dilewati')
        continue
    if glob.glob(f'{d}/*.npz'):
        print(f'ccc/{ds}: .npz ada di Drive, ekstrak ulang tanpa mengunduh')
    else:
        print(f'ccc/{ds}: mengunduh (beberapa GB, sabar)...')
    if not glob.glob(f'{d}/*.npz'):
        r = subprocess.run(['gdown', '--fuzzy', GID_SCORES_CCC[ds]],
                           cwd=d, capture_output=True, text=True)
        if r.stdout.strip():
            print('  stdout:', r.stdout.strip()[-500:])
        if r.stderr.strip():
            print('  stderr:', r.stderr.strip()[-300:])
        print(f'  returncode: {r.returncode}  <- BUKAN bukti; diverifikasi di bawah')

    for p in sorted(glob.glob(f'{d}/*')):
        low = p.lower()
        if low.endswith('.zip'):
            subprocess.run(['unzip','-o','-q',p,'-d',d], check=False)
        elif low.endswith(('.tar.gz','.tgz','.tar')):
            subprocess.run(['tar','-xf',p,'-C',d], check=False)

    # .npz ADALAH zip berisi beberapa .npy. Versi sebelumnya mencocokkan ekstensi
    # secara literal ('.zip'/'.tar'), jadi imagenet.npz dilewati dan tidak ada .npy
    # terbentuk -- padahal unduhannya berhasil penuh. Satu baris, kegagalan bisu.
    #
    # Header .npy dibaca lewat zipfile TANPA mendekompresi isinya, supaya bentuk dan
    # dtype tiap anggota terlihat tanpa memuat gigabyte. Lalu HANYA dua array yang
    # dibutuhkan dimaterialkan, dan ke /content (ephemeral) bukan Drive -- mengekstrak
    # seluruh 4,6 GB ke Drive akan menggandakan pemakaian kuota tanpa alasan.
    for p in sorted(glob.glob(f'{d}/*.npz')):
        print(f'  membaca header {os.path.basename(p)} (tanpa dekompresi)...')
        members = []
        with zipfile.ZipFile(p) as zf:
            for nm in zf.namelist():
                try:
                    with zf.open(nm) as fh:
                        ver = np.lib.format.read_magic(fh)
                        if ver == (1, 0):
                            shp, _fo, dt = np.lib.format.read_array_header_1_0(fh)
                        elif ver == (2, 0):
                            shp, _fo, dt = np.lib.format.read_array_header_2_0(fh)
                        else:
                            continue
                    members.append((nm, shp, dt))
                    print(f'    {nm:36s} {str(shp):20s} {dt}')
                except Exception as e:
                    print(f'    {nm:36s} header tak terbaca: {type(e).__name__}')

        twod = [m for m in members if len(m[1]) == 2]
        oned = [m for m in members if len(m[1]) == 1]
        if not twod or not oned:
            print('  npz ini tidak memuat pasangan (2-D, 1-D) — kirim daftar di atas.')
            continue
        sc = max(twod, key=lambda m: m[1][0] * m[1][1])
        lb = next((m for m in oned if m[1][0] == sc[1][0]), None)
        if lb is None:
            print(f'  tidak ada array 1-D sepanjang {sc[1][0]} untuk mendampingi {sc[0]}')
            continue
        out = f'/content/ccc_npy/{ds}'
        os.makedirs(out, exist_ok=True)
        print(f'  materialkan {sc[0]} {sc[1]} dan {lb[0]} {lb[1]} -> {out}')
        # zipfile.namelist() memberi nama DENGAN sufiks '.npy', tetapi NpzFile
        # diindeks TANPA sufiks -> z['softmax.npy'] KeyError, z['softmax'] benar.
        # Ditemukan oleh tes sintetik sebelum run nyata.
        k_sc = sc[0][:-4] if sc[0].endswith('.npy') else sc[0]
        k_lb = lb[0][:-4] if lb[0].endswith('.npy') else lb[0]
        with np.load(p, allow_pickle=False) as z:
            np.save(f'{out}/scores.npy', z[k_sc])
            np.save(f'{out}/labels.npy', z[k_lb])

    files = inventory(d, f'ccc/{ds}')
    npys = (glob.glob(f'{d}/**/*.npy', recursive=True)
            + glob.glob(f'/content/ccc_npy/{ds}/*.npy'))
    if not npys:
        print(f'  GAGAL: tidak ada .npy terbentuk untuk ccc/{ds}.')
        for p in files:
            is_html, head = looks_like_html(p)
            if is_html:
                print(f'  PENYEBAB: {os.path.basename(p)} adalah HALAMAN HTML, bukan data.')
                print('  Itu batas kuota Google Drive; gdown tetap keluar dengan kode 0.')
                print('  cuplikan:', ' '.join(head.decode('utf-8','replace').split())[:300])
                print(f'  URL: https://drive.google.com/uc?id={GID_SCORES_CCC[ds]}')
                break
        else:
            print('  Berkas ADA tetapi tidak menghasilkan .npy — kirim daftar di atas.')
    else:
        print(f'  OK: {len(npys)} .npy siap dipakai')

# dua akar untuk CCC: Drive (kalau .npy langsung) dan /content (hasil materialisasi
# anggota .npz). Keduanya diperiksa supaya tidak peduli bentuk rilisnya.
roots = ([(ltc_dir(ds), 'ltc', ds) for ds in LTC_DATASETS]
         + [(ccc_dir(ds), 'ccc', ds) for ds in CCC_DATASETS]
         + [(f'/content/ccc_npy/{ds}', 'ccc', ds) for ds in CCC_DATASETS])
found = []
for rt, src, ds in roots:
    for f in sorted(glob.glob(f'{rt}/**/*.npy', recursive=True)):
        found.append((f, src, ds))
print()
print(f'total .npy: {len(found)}')
for f, src, ds in found[:60]:
    a = np.load(f, mmap_mode='r')
    print(f'  [{src}] {os.path.basename(f):52s} {str(a.shape):18s} {a.dtype}')
if not found:
    print('TIDAK ADA .npy sama sekali. Isi direktori mentah:')
    for rt, src, ds in roots:
        for p in sorted(glob.glob(f'{rt}/**/*', recursive=True))[:25]:
            if os.path.isfile(p):
                print(f'  {os.path.relpath(p, DRIVE_ROOT):64s} {os.path.getsize(p)/1e6:8.1f} MB')


### 3b. Referensi — dari mana ID CCC berasal (opsional, tidak perlu dijalankan)

ID di sel 1 diambil dari `download_data.sh` milik CCC. Sel ini hanya untuk memverifikasi
bahwa ID-nya belum berubah; ia **tidak diperlukan** untuk menjalankan notebook.


In [ ]:
SHOW_CCC_SCRIPT = False
if SHOW_CCC_SCRIPT:
    if not os.path.isdir('/content/ccc'):
        subprocess.run(['git','clone','--depth','1',
                        'https://github.com/tiffanyding/class-conditional-conformal.git',
                        '/content/ccc'], check=False)
    print(open('/content/ccc/download_data.sh').read())
    print('bandingkan dengan GID_SCORES_CCC di sel 1')
else:
    print('dilewati (ID sudah ditanam di sel 1)')


## 4. SURVEI semua dump — mana yang punya daya uji?

Menjawab pertanyaan dataset secara **empiris**. Yang menentukan daya uji gate C bukan jumlah
kelas, melainkan **jumlah kelas yang punya cukup sampel KALIBRASI**.

Pasangan skor/label dicocokkan lewat **konvensi nama** LTC (`..._softmax.npy` ↔
`..._labels.npy`) — percobaan pertama mencocokkan lewat dimensi array dan gagal menemukan
apa pun tanpa memberi tahu mengapa.


In [ ]:
import numpy as np
from pcc.data.load import softmax_from_logits

NEED_TOTAL = int(np.ceil(N_CAL_PRIMARY / FRAC_CAL))   # n_cal harus tercapai DI porsi CAL

def is_focal(p):
    return 'focal' in p.replace(chr(92), '/').lower()

def variant_ok(p):
    return is_focal(p) if LOSS_VARIANT == 'focal' else (not is_focal(p))

# PENCOCOKAN PASANGAN. Dua konvensi, dan KEDUANYA selalu dijalankan:
#   LTC : *_softmax.npy  <-> *_labels.npy   (nama eksplisit)
#   CCC : nama bebas; (2-D, 1-D) dengan jumlah baris sama di direktori sama
#
# Versi sebelumnya menjalankan yang generik sebagai FALLBACK ('if not pairs'). Karena
# dump LTC selalu menghasilkan pasangan, fallback itu tidak pernah jalan dan berkas CCC
# diabaikan tanpa suara -- tabel survei kehilangan baris ccc sepenuhnya. Either/or di
# tempat yang seharusnya both.
pairs, why, seen = [], [], set()

for f, _src, _ds in found:
    if '_softmax.npy' not in f:
        continue
    lab = f.replace('_softmax.npy', '_labels.npy')
    if os.path.exists(lab):
        pairs.append((f, lab, 'nama', _src, _ds))
        seen.add(f)
    else:
        why.append(f'{os.path.basename(f)}: label _labels.npy pendamping tidak ada')

for f2, _src, _ds in found:
    if f2 in seen:
        continue
    a = np.load(f2, mmap_mode='r')
    if a.ndim != 2:
        continue
    mate = None
    for f1, _s1, _d1 in found:
        if f1 == f2 or os.path.dirname(f1) != os.path.dirname(f2):
            continue
        b = np.load(f1, mmap_mode='r')
        if b.ndim == 1 and len(b) == len(a):
            mate = f1
            break
    if mate is not None:
        pairs.append((f2, mate, 'dimensi', _src, _ds))
        seen.add(f2)
    else:
        why.append(f'{os.path.basename(f2)}: tidak ada array 1-D sepanjang {len(a)} '
                   f'di direktori yang sama')

for w in why[:20]:
    print('  ', w)
print(f'  pasangan ditemukan: {len(pairs)} '
      f"(nama: {sum(1 for p in pairs if p[2] == 'nama')}, "
      f"dimensi: {sum(1 for p in pairs if p[2] == 'dimensi')})")
if not pairs:
    print()
    print('TIDAK ADA pasangan skor/label. Semua .npy yang terlihat:')
    for f, _s, _d in found:
        a = np.load(f, mmap_mode='r')
        print(f'   [{_s}] {os.path.relpath(f, DRIVE_ROOT):58s} {str(a.shape):18s} {a.dtype}')
    print()
    print('Kirim daftar ini — konvensinya perlu disesuaikan, jangan diteruskan menebak.')

cands = []
for f2, f1, how, src, ds in pairs:
    S = np.load(f2, mmap_mode='r'); y = np.load(f1).astype(int)
    if S.ndim != 2 or len(y) != len(S) or y.min() < 0 or y.max() >= S.shape[1]:
        print(f'  bentuk tidak konsisten: {os.path.basename(f2)} {S.shape} vs label {y.shape}')
        continue
    c = np.bincount(y, minlength=S.shape[1])
    cands.append(dict(scores=f2, labels=f1, n=len(y), K=int(S.shape[1]),
                      med=float(np.median(c)), empty=int((c == 0).sum()),
                      n_feasible=int((c >= NEED_TOTAL).sum()),
                      focal=is_focal(f2), keep=variant_ok(f2), how=how,
                      src=src, ds=ds))

assert cands, 'tidak ada pasangan yang bisa dipakai — lihat daftar di atas'
hdr = ('src'.ljust(5) + 'dump'.ljust(46) + 'n'.rjust(9) + 'K'.rjust(7)
       + 'med/kls'.rjust(9) + 'layak'.rjust(8) + '  varian')
print(hdr)
for r in sorted(cands, key=lambda r: (-r['keep'], -r['n_feasible'])):
    nm = os.path.basename(r['scores'])[-44:]
    v = ('focal' if r['focal'] else 'cross_entropy') + ('' if r['keep'] else '  [DIABAIKAN]')
    print(r['src'].ljust(5) + nm.ljust(46) + str(r['n']).rjust(9) + str(r['K']).rjust(7)
          + f"{r['med']:9.0f}" + str(r['n_feasible']).rjust(8) + '  ' + v)

usable = [r for r in cands if r['keep']]
assert usable, f'tidak ada dump varian {LOSS_VARIANT} — cek LOSS_VARIANT'
best = max(usable, key=lambda r: r['n_feasible'])
PREMISE_OK = best['n_feasible'] >= 500
print()
print('terbaik:', os.path.relpath(best['scores'], DRIVE_ROOT))
print(f"  {best['n_feasible']} kelas punya >= {NEED_TOTAL} sampel "
      f"(agar {N_CAL_PRIMARY} tercapai di porsi CAL {FRAC_CAL:.0%})")
print('PREMIS PRE-DEKLARASI (>=500 kelas layak):',
      'TERPENUHI' if PREMISE_OK else 'TIDAK TERPENUHI')
if not PREMISE_OK:
    print()
    print('  TEMUAN: tidak ada dump LTC yang memberi daya uji yang dibutuhkan.')
    print('  Artinya batas 38-kelas/stratum di Pl@ntNet BUKAN kekhususan Pl@ntNet —')
    print('  ia berlaku untuk keluarga dataset ekor-panjang yang dipakai prior art.')
    print('  JANGAN tafsirkan gate B/C dari run ini. Tambah dump berkelas-banyak')
    print("  ke CCC_DATASETS di sel 1 (mis. 'inaturalist' = iNat-2021, 633 kelas).")


# BERHENTI DI SINI, bukan lanjut. Membiarkan run jalan dengan premis yang gagal
# memindahkan kegagalan ke sel 7 (screen stabilitas), di mana ia terlihat seperti
# deskriptor-tidak-stabil padahal sebabnya sampel-per-kelas-tidak-cukup. Dua
# diagnosis yang sangat berbeda, dan yang kedua bukan kegagalan metode.
_nf = best['n_feasible']
_nm = os.path.basename(best['scores'])
assert PREMISE_OK, (
    'PREMIS GAGAL. Dump terbaik ' + _nm + ' hanya punya ' + str(_nf) +
    ' kelas dengan >= ' + str(NEED_TOTAL) + ' sampel; butuh >= 500. '
    'Ini TEMUAN, bukan error: dump LTC tidak punya daya uji yang dibutuhkan. '
    'Tambah dump berkelas-banyak ke CCC_DATASETS di sel 1 lalu ulangi.')

# float32: dump besar x salinan thr_lac bisa beberapa GB; sesi ini pernah crash karena
# alokasi besar. float64 hanya perlu saat mereproduksi softmax rilis bit-per-bit.
# mmap dulu: jangan pernah memuat 4,6 GB hanya untuk membuang 80%-nya.
S_mm = np.load(best['scores'], mmap_mode='r')
y_full = np.load(best['labels']).astype(int)
K = S_mm.shape[1]
cnt_full = np.bincount(y_full, minlength=K)   # prevalensi SEJATI, sebelum subsample

if MAX_ROWS is not None and len(y_full) > MAX_ROWS:
    keep_frac = MAX_ROWS / len(y_full)
    rr = np.random.default_rng(SEED)
    sel = []
    for c in range(K):
        idx = np.where(y_full == c)[0]
        if len(idx) == 0:
            continue
        k = max(1, int(round(keep_frac * len(idx))))
        sel.append(rr.choice(idx, k, replace=False) if k < len(idx) else idx)
    sel = np.sort(np.concatenate(sel))
    print(f'subsample terstratifikasi: {len(y_full)} -> {len(sel)} baris '
          f'({keep_frac:.1%} per kelas, struktur prevalensi dipertahankan)')
else:
    sel = np.arange(len(y_full))
    print(f'dump dipakai utuh: {len(sel)} baris')

# float32: cukup untuk kuantil probabilitas. float64 hanya perlu saat mereproduksi
# softmax rilis bit-per-bit, yang tidak dilakukan di sini.
S_all = np.ascontiguousarray(S_mm[sel]).astype(np.float32, copy=False)
y_all = y_full[sel]
del S_mm
if N_CLASSES_EXPECTED is not None:
    assert K == N_CLASSES_EXPECTED, f'kelas {K}, diharapkan {N_CLASSES_EXPECTED}'
print()
print(f'dipakai: {S_all.shape} {S_all.dtype} ({S_all.nbytes/1e6:.0f} MB dalam RAM)')
print(f'prevalensi sejati: min {cnt_full.min()} median {np.median(cnt_full):.0f} '
      f'max {cnt_full.max()}  -> log-range {np.ptp(np.log(np.maximum(cnt_full,1))):.3f}')

row = S_all[:200].sum(axis=1)
is_prob = bool(np.allclose(row, 1.0, atol=1e-3))
print(f'baris berjumlah 1 (sudah softmax): {is_prob}')
if not is_prob:
    print('  -> diperlakukan sebagai logits, dikonversi')
    S_all = softmax_from_logits(S_all)
cnt = np.bincount(y_all, minlength=K)
acc = float((S_all.argmax(axis=1) == y_all).mean())
print(f'akurasi top-1: {acc:.4f} | kelas kosong: {(cnt==0).sum()}')


## 5. Tiga split terpisah — leak guard ditegakkan

DESC 40% (phi saja) / CAL 30% (delta_y) / EVAL 30% (semua metrik). Stratified per kelas.


In [ ]:
rng = np.random.default_rng(SEED)
role = np.empty(len(y_all), dtype='<U4')
for c in range(K):
    idx = np.where(y_all == c)[0]
    rng.shuffle(idx)
    n = len(idx); n_d = int(round(FRAC_DESC*n)); n_c = int(round(FRAC_CAL*n))
    role[idx[:n_d]] = 'desc'
    role[idx[n_d:n_d+n_c]] = 'cal'
    role[idx[n_d+n_c:]] = 'eval'

i_desc = np.where(role == 'desc')[0]
i_cal  = np.where(role == 'cal')[0]
i_eval = np.where(role == 'eval')[0]
assert len(set(i_desc) & set(i_cal)) == 0
assert len(set(i_cal) & set(i_eval)) == 0
assert len(set(i_desc) & set(i_eval)) == 0
assert len(i_desc) + len(i_cal) + len(i_eval) == len(y_all)
print(f'DESC {len(i_desc)}  CAL {len(i_cal)}  EVAL {len(i_eval)}  (terpisah, terverifikasi)')

cnt_cal = np.bincount(y_all[i_cal], minlength=K)
cnt_desc = np.bincount(y_all[i_desc], minlength=K)
print(f'CAL sampel/kelas: median {np.median(cnt_cal):.0f} min {cnt_cal.min()}')
print(f'kelas dengan >= {N_CAL_PRIMARY} sampel CAL: {(cnt_cal >= N_CAL_PRIMARY).sum()}/{K}')


## 6. phi(y) dari ruang OUTPUT — hanya dari split DESC


In [ ]:
from pcc.descriptors.output_space import build_output_descriptors, QUOTA_DETERMINED

# log_prevalence dari hitungan SEJATI seluruh dump, bukan dari hitungan DESC. Dengan
# subsampling, hitungan DESC hanyalah kuota split -- memakainya membuat ablasi
# prevalensi menguji kuota kita sendiri, bukan prevalensi kelas.
Phi, names = build_output_descriptors(S_all[i_desc], y_all[i_desc], K,
                                     log_prevalence_from=cnt_full)
print(f'Phi {Phi.shape} | baris finite: {int(np.isfinite(Phi).all(axis=1).sum())}/{K}')
print('fitur:', names)
print('QUOTA_DETERMINED (tidak boleh dikreditkan stabilitas):', QUOTA_DETERMINED)


## 7. Screen stabilitas — set PRIMER = fitur dengan stabilitas >= 0,90

Dihitung dengan membelah split DESC jadi dua bagian terpisah dan mengorelasikan phi lintas
kelas. Prosedurnya sama seperti Pl@ntNet, jadi ambangnya bisa dibandingkan.


In [ ]:
half = {}
r2 = np.random.default_rng(SEED + 7)
mask = np.zeros(len(i_desc), bool)
for c in range(K):
    loc = np.where(y_all[i_desc] == c)[0]
    r2.shuffle(loc)
    mask[loc[:len(loc)//2]] = True
PhiA, _ = build_output_descriptors(S_all[i_desc][mask],  y_all[i_desc][mask],  K,
                                  log_prevalence_from=cnt_full)
PhiB, _ = build_output_descriptors(S_all[i_desc][~mask], y_all[i_desc][~mask], K,
                                  log_prevalence_from=cnt_full)
stab = {}
for j, nm in enumerate(names):
    a, b = PhiA[:, j], PhiB[:, j]
    ok = np.isfinite(a) & np.isfinite(b)
    stab[nm] = (float(np.corrcoef(a[ok], b[ok])[0, 1])
                if ok.sum() > 3 and np.std(a[ok]) > 0 and np.std(b[ok]) > 0 else np.nan)
for nm in sorted(stab, key=lambda n: -(stab[n] if np.isfinite(stab[n]) else -9)):
    tag = ' [QUOTA]' if nm in QUOTA_DETERMINED else ''
    print(f'  {nm:18s} {stab[nm]:+.3f}{tag}')

# Stabilitas rendah punya DUA sebab yang harus dipisahkan: deskriptor yang memang
# berderau, atau sampel per kelas yang tidak cukup untuk mengestimasinya. Screen ini
# membelah DESC jadi dua, jadi yang tersedia per kelas adalah setengah dari DESC.
cA = np.bincount(y_all[i_desc][mask], minlength=K)
cB = np.bincount(y_all[i_desc][~mask], minlength=K)
print()
print(f'sampel/kelas yang tersedia untuk screen: median {np.median(cA):.1f} per separuh '
      f'(min {cA.min()}, kelas dengan 0: {(cA==0).sum()}/{K})')
if np.median(cA) < 5:
    print('  PERINGATAN: <5 sampel/kelas per separuh. Stabilitas rendah di sini TIDAK')
    print('  bisa dibaca sebagai deskriptor yang buruk — ia tidak dapat diestimasi.')

stable_names = [n for n in names
                if n not in QUOTA_DETERMINED
                and np.isfinite(stab[n]) and stab[n] >= STABLE_THRESHOLD]
print()
print(f'lolos screen ({len(stable_names)}):', stable_names)

# prof_knn_1 DICADANGKAN sebagai baseline jarak pre-registered dan DIKELUARKAN dari
# model penuh. Kalau ia ikut di dalam model, gate C hanya menguji apakah fitur sisanya
# menambah sesuatu di atas prof_knn_1 — submodel bersarang, bukan perbandingan
# terhadap baseline prior-art gaya Fargion yang dimaksud Sec 6.5C.
DISTANCE_BASELINE = 'prof_knn_1'
stable_names = [n for n in stable_names if n != DISTANCE_BASELINE]
full_names = [n for n in names if n != DISTANCE_BASELINE]
print(f'baseline jarak dicadangkan: {DISTANCE_BASELINE} (di luar model penuh)')
print(f'set PRIMER stable ({len(stable_names)}):', stable_names)
print('Sec 3.3 terpenuhi:', bool(stable_names))
_peak = max([v for n, v in stab.items()
             if n not in QUOTA_DETERMINED and np.isfinite(v)], default=0.0)
assert stable_names, (
    'tidak ada fitur lolos screen >= ' + str(STABLE_THRESHOLD) +
    '. Puncak: ' + format(_peak, '.3f') + '. JANGAN turunkan ambang. '
    'Periksa dulu sampel/kelas yang dilaporkan di atas: kalau < 5 per separuh, '
    'sebabnya jumlah sampel, bukan kualitas deskriptor.')

lp = Phi[:, names.index('log_prevalence')]
lp = lp[np.isfinite(lp)]
vac = np.std(lp) < 1e-8
print(f'log_prevalence: sd={np.std(lp):.6f} rentang={np.ptp(lp):.6f}')
print('  ablasi prevalensi:', 'HAMPA (dikeluarkan dari verdict)' if vac else 'bisa diuji')
FEATURE_SETS = {'stable': stable_names, 'full': full_names}


## 8. Gate A — reliabilitas delta_y, dan plafonnya


In [ ]:
from pcc.scores.base import thr_lac
from pcc.targets.delta import split_half_reliability, delta_y_matched_n
from pcc.eval.stats import mean_ci

S_cal = thr_lac(S_all[i_cal]); y_cal = y_all[i_cal]
s_true_cal = S_cal[np.arange(len(y_cal)), y_cal]
S_ev = thr_lac(S_all[i_eval]); y_ev = y_all[i_eval]

relA = split_half_reliability(s_true_cal, y_cal, K, ALPHA_PRIMARY,
                              n_splits=N_SPLITS_A, seed=SEED)
r_delta = float(relA['reliability_mean'])
ciA = mean_ci(relA['reliability_splits'])
print(f"gate A r_delta = {r_delta:.3f}  CI [{ciA['ci_low']:.3f}, {ciA['ci_high']:.3f}]")
print(f"  kelas eligible: {relA['n_classes_eligible']}  "
      f"split bernilai: {relA['n_splits_with_a_value']}/{N_SPLITS_A}")
gate_A_pass = bool(np.isfinite(ciA['ci_low']) and ciA['ci_low'] >= 0.30)
print('gate A:', 'LULUS' if gate_A_pass else 'GAGAL')

r_phi = float(np.nanmean([stab[n] for n in stable_names]))
print(f'r_phi (rata-rata stabilitas set stable) = {r_phi:.3f}')
print(f'plafon gabungan r_delta*r_phi = {r_delta*r_phi:.3f}')


## 9. delta_y pada n_cal tercocokkan + null prevalensi

ImageNet berimbang, jadi `log n_y` nyaris konstan dan null prevalensi kemungkinan **tidak
terdefinisi** — itu diharapkan dan dilaporkan, bukan error.


In [ ]:
from pcc.targets.delta import prevalence_null

deltas = {}
for tag, ncal in (('primary', N_CAL_PRIMARY), ('secondary', N_CAL_SECONDARY)):
    d, kept = delta_y_matched_n(s_true_cal, y_cal, K, ALPHA_PRIMARY, n_cal=ncal, seed=SEED)
    deltas[tag] = (d, kept, ncal)
    nz = int(np.isfinite(d).sum())
    print(f'[{tag}] n_cal={ncal}: delta_y terdefinisi untuk {nz}/{K} kelas '
          f'(sd {np.nanstd(d):.4f})')

pn = prevalence_null(s_true_cal, y_cal, K, ALPHA_PRIMARY, n_cal=N_CAL_PRIMARY,
                     n_reps=30, seed=SEED)
print('null prevalensi:', pn.get('undefined_reason') or
      f"mean {pn['null_mean']:+.3f} sd {pn['null_sd']:.3f}")


## 10. UJI PRIMER — gate B (bootstrap KELAS) dan gate C (permutasi KELAS + Holm)

Kriteria yang lama, *CI selisih mengecualikan 0* atas sebaran antar-split, **bukan uji yang
valid**: nol bukan nilai null-nya (terukur: 38 kelas tanpa sinyal memberi selisih +0,109
sementara null berpusat di -0,102), dan split adalah pemakaian ulang kelas yang sama, bukan
observasi independen. Unit yang bisa ditukar adalah **kelas**.


In [ ]:
from pcc.eval.predictability import (predictability, predictability_class_bootstrap,
                                     class_permutation_p)
from pcc.eval.stats import holm_bonferroni

d_prim = deltas['primary'][0]
feats = FEATURE_SETS['stable']

boot = predictability_class_bootstrap(Phi, d_prim, names, feature_subset=feats,
                                     n_boot=N_BOOT_CLASS, n_splits=10, seed=SEED,
                                     reliability=r_delta)
print('GATE B — bootstrap tingkat-KELAS (PRIMER)')
print(f"  R2 {boot['mean']:+.4f}  CI [{boot['ci_low']:+.4f}, {boot['ci_high']:+.4f}]"
      f"  n_kelas={boot['n_classes']} n_boot={boot['n_boot']}")
if 'normalized_mean' in boot:
    print(f"  ter-normalisasi {boot['normalized_mean']:+.4f} "
          f"CI [{boot['normalized_ci_low']:+.4f}, {boot['normalized_ci_high']:+.4f}]")
gate_B_pass = bool(boot.get('gate_B_pass_class_unit'))
print('  gate B:', 'LULUS' if gate_B_pass else 'GAGAL')

spl = predictability(Phi, d_prim, names, feature_subset=feats, reliability=r_delta,
                     n_splits=N_SPLITS_BC, seed=SEED)
sf = spl['r2_by_predictor']['full']
print(f"  [sekunder, CI antar-split] R2 {sf['mean']:+.4f} "
      f"[{sf['ci_low']:+.4f}, {sf['ci_high']:+.4f}]")
print(f"  baseline jarak: {spl['distance_col_used']} "
      f"(bersarang di full: {spl['distance_baseline_is_nested_in_full']})")


In [ ]:
# Keluarga uji PRIMER. Ablasi prevalensi hanya masuk kalau ia BUKAN hampa —
# memasukkan uji yang dijamin menang akan MENGENCERKAN koreksi Holm dan membuat
# gate C lebih mudah, bukan lebih ketat.
ABLATIONS_PRIMARY = ['distance_only']
if not spl['prevalence_ablation_degenerate']:
    ABLATIONS_PRIMARY.append('log_prevalence_only')
else:
    print('CATATAN: ablasi prevalensi HAMPA (dataset berimbang) — dikeluarkan dari')
    print('  keluarga uji primer. Gate C di sini menguji HANYA lawan baseline jarak:')
    print('  lengan prior-art yang lebih penting, tetapi uji yang LEBIH SEMPIT')
    print('  daripada di Pl@ntNet, dan harus dilaporkan sebagai lebih sempit.')
print(f'GATE C — null permutasi tingkat-KELAS, n_perm={N_PERM_CLASS} (PRIMER)')
perm_res, pvals, labels = {}, [], []
for abl in ABLATIONS_PRIMARY:
    if abl not in spl['gate_C_detail']:
        print(f'  {abl}: tidak tersedia di set fitur ini — dilewati')
        continue
    r = class_permutation_p(Phi, d_prim, names, feature_subset=feats, ablation=abl,
                            n_perm=N_PERM_CLASS, n_splits=20, seed=SEED)
    perm_res[abl] = r
    pvals.append(r['p_value']); labels.append(abl)
    print(f"  {abl:24s} obs {r['observed']:+.4f} null {r['null_mean']:+.4f}"
          f" (sd {r['null_sd']:.4f})  p={r['p_value']:.4f}")

holm = holm_bonferroni(pvals, alpha=0.05) if pvals else None
gate_C_pass = False
if holm:
    print()
    print('  Holm-Bonferroni lintas keluarga uji (Sec 8.6):')
    for k, h in zip(labels, holm):
        pv = h['p_value']; th = h['threshold']; rj = h['reject']
        print(f'    {k:24s} p={pv:.4f} ambang={th:.4f} tolak_H0={rj}')
    gate_C_pass = all(h['reject'] for h in holm)
print('  gate C:', 'LULUS' if gate_C_pass else 'GAGAL')


## 11. SEKUNDER — multi-alpha (Sec 8.8), yang Pl@ntNet tidak pernah bisa

Dengan ~115 sampel/kelas, alpha=0,01 butuh >=99 dan **terpenuhi**. Di Pl@ntNet alpha=0,01
hanya layak untuk 57 dari 1.081 kelas.


In [ ]:
multi = {}
for a in (ALPHA_PRIMARY,) + tuple(ALPHAS_SECONDARY):
    need = int(np.ceil(1/a)) - 1
    feasible = int((cnt_cal >= need).sum())
    st = thr_lac(S_all[i_cal])[np.arange(len(y_cal)), y_cal]
    d_a, _ = delta_y_matched_n(st, y_cal, K, a, n_cal=N_CAL_PRIMARY, seed=SEED)
    nz = int(np.isfinite(d_a).sum())
    if nz < 50:
        multi[a] = {'skipped': True, 'n_classes': nz}
        print(f'alpha={a}: hanya {nz} kelas — dilewati')
        continue
    b = predictability_class_bootstrap(Phi, d_a, names, feature_subset=feats,
                                       n_boot=200, n_splits=8, seed=SEED,
                                       reliability=r_delta)
    # Kuantil conformal butuh n >= ceil(1/a)-1 DI DALAM n_cal yang dipakai. Kalau tidak,
    # estimator empiris tetap mengembalikan angka -- maksimum sampelnya -- tetapi itu
    # BUKAN kuantil (1-a), dan R2-nya mengukur hal lain. Ditandai, bukan dilaporkan
    # berdampingan seolah setara.
    meaningful = bool(N_CAL_PRIMARY >= need)
    multi[a] = {'n_classes': nz, 'feasible_classes': feasible,
                'n_needed_for_alpha': need, 'meaningful': meaningful,
                'r2': b['mean'], 'ci': [b['ci_low'], b['ci_high']],
                'gate_B': bool(b.get('gate_B_pass_class_unit'))}
    tag = ' <- PRIMER' if a == ALPHA_PRIMARY else ''
    if not meaningful:
        tag += f'  [TIDAK BERMAKNA: butuh n>={need}, n_cal={N_CAL_PRIMARY}]'
    print(f"alpha={a}: kelas layak {feasible}/{K}, delta_y {nz}, "
          f"R2 {b['mean']:+.4f} [{b['ci_low']:+.4f},{b['ci_high']:+.4f}] "
          f"gate_B={multi[a]['gate_B']}{tag}")


## 12. SEKUNDER — set `full`, n_cal=50, dan Sec 6.4 (Amandemen 8)


In [ ]:
sec = {}
for fset in ('stable', 'full'):
    b = predictability_class_bootstrap(Phi, d_prim, names,
                                       feature_subset=FEATURE_SETS[fset],
                                       n_boot=200, n_splits=8, seed=SEED,
                                       reliability=r_delta)
    sec[f'{fset}|n_cal{N_CAL_PRIMARY}'] = b
    print(f"[{fset}] p={len(FEATURE_SETS[fset])} R2 {b['mean']:+.4f} "
          f"[{b['ci_low']:+.4f},{b['ci_high']:+.4f}] gate_B={b.get('gate_B_pass_class_unit')}")

d_sec = deltas['secondary'][0]
b2 = predictability_class_bootstrap(Phi, d_sec, names, feature_subset=feats,
                                    n_boot=200, n_splits=8, seed=SEED,
                                    reliability=r_delta)
sec[f'stable|n_cal{N_CAL_SECONDARY}'] = b2
print(f"[stable, n_cal={N_CAL_SECONDARY}] R2 {b2['mean']:+.4f} "
      f"[{b2['ci_low']:+.4f},{b2['ci_high']:+.4f}] gate_B={b2.get('gate_B_pass_class_unit')}")


In [ ]:
from pcc.eval.predictability import ridge_fit, ridge_predict
from pcc.eval.setsize import setsize_translation_shrunk
from pcc.eval.decomposition import group_quantile

cols = [names.index(f) for f in feats]
usable = np.where(np.isfinite(d_prim) & np.isfinite(Phi[:, cols]).all(axis=1))[0]
print(f'kelas usable untuk Sec 6.4: {len(usable)}')
qg = group_quantile(s_true_cal, ALPHA_PRIMARY, 'empirical')
rg = np.random.default_rng(SEED)
acc64 = {'obs': [], 'null': [], 'oracle': [], 'raw': [], 'lam': []}
for rep in range(20):
    perm = rg.permutation(usable)
    fit_c, held_c = perm[:len(perm)//2], perm[len(perm)//2:]
    m = ridge_fit(Phi[fit_c][:, cols], d_prim[fit_c], 1.0)
    dh = np.zeros(K)
    dh[held_c] = ridge_predict(m, Phi[held_c][:, cols])
    dh[fit_c]  = ridge_predict(m, Phi[fit_c][:, cols])
    dn = np.array(dh); dn[held_c] = rg.permutation(dh[held_c])
    for tag, dd in (('obs', dh), ('null', dn)):
        try:
            r = setsize_translation_shrunk(S_ev, y_ev, ALPHA_PRIMARY, None, None,
                                           fit_c, held_c, dd, stat='worst', q_global=qg)
        except ValueError:
            continue
        acc64[tag].append(r['delta']['worst'])
        if tag == 'obs':
            acc64['lam'].append(r['lambda_selected_on_train'])
            acc64['oracle'].append(r['controls']['oracle_ceiling'])
            acc64['raw'].append(r['controls']['raw_delta_lambda1'])
o = mean_ci(np.array(acc64['obs'], float)); n0 = mean_ci(np.array(acc64['null'], float))
orc = mean_ci(np.array(acc64['oracle'], float)); raw = mean_ci(np.array(acc64['raw'], float))
sec64 = {'observed': o, 'shuffled_null': n0, 'oracle_ceiling': orc,
         'raw_delta_lambda1': raw,
         'lambda_mean': float(np.mean(acc64['lam'])) if acc64['lam'] else float('nan'),
         'beats_null': bool(o['ci_low'] > n0['ci_high']),
         'pass': bool(o['ci_low'] > 0 and o['ci_low'] > n0['ci_high'])}
print(f"Sec 6.4: observed {o['mean']:+.4f} [{o['ci_low']:+.4f},{o['ci_high']:+.4f}]")
print(f"  null {n0['mean']:+.4f} | ORACLE {orc['mean']:+.4f} | raw lam=1 {raw['mean']:+.4f}")
print(f"  lambda dari TRAIN {sec64['lambda_mean']:.3f} -> "
      f"{'LULUS' if sec64['pass'] else 'TIDAK POSITIF'}")
if orc['mean'] <= 0.02:
    print('  PERINGATAN: tanpa ruang oracle, metriknya tidak bisa positif — jangan dibaca')


## 12b. UJI NON-SIRKULAR — phi dari bobot kepala klasifier (`w_y`)

Hasil ruang-output lolos gate B/C, tetapi **tidak dapat menegakkan klaim geometri**, dan
alasannya struktural: phi di sana dibangun dari matriks skor, dan delta_y juga. `conf_mean`
pada DESC dan `q_y` pada CAL adalah dua estimasi distribusi skor kelas yang SAMA. Terukur:
baseline jarak sendirian hanya menjelaskan ~0,155 dari R2 0,497 — sisanya ringkasan skor.

`w_y` mematahkan sirkularitas itu pada tiga hal:

1. **Parameter, bukan estimasi sampel** — derau nol, tersedia untuk setiap kelas tanpa satu
   pun sampel berlabel. Itu properti yang justru dibutuhkan klaim ekstrapolasi.
2. **Tanpa citra, tanpa GPU** — matriks `(K, d)` dari checkpoint, unduhan ~100 MB.
3. **Dari model yang BERBEDA** dengan yang menghasilkan skornya: skor CCC berasal dari
   SimCLRv2 + linear probe, sementara kepala ini ResNet-50 tersupervisi torchvision. Jadi phi
   **eksogen** terhadap delta_y — inilah yang membuat ujinya bermakna.

Konsekuensi yang harus dinyatakan: stabilitas fitur `w_y` adalah **1,0 secara konstruksi**,
bukan diperoleh. Jadi screen stabilitas tidak berlaku di sini dan `r_phi` **tidak boleh**
diambil 1,0 lalu dipakai menormalkan — plafonnya menjadi `r_delta` saja.


In [ ]:
head = None
try:
    from pcc.descriptors.head_weights import (build_head_descriptors,
                                              load_torchvision_resnet50_head,
                                              EXACT_BY_CONSTRUCTION)
    W_head, b_head = load_torchvision_resnet50_head()
    nb = 0 if b_head is None else len(b_head)
    print('kepala torchvision ResNet-50: W', W_head.shape, 'b', nb)
    assert W_head.shape[0] == K, (
        'kepala punya ' + str(W_head.shape[0]) + ' kelas, dump punya ' + str(K) +
        ' -- indeks kelas tidak sebanding, JANGAN dipakai')

    Phi_h, names_h = build_head_descriptors(W_head, b_head)
    # log_prevalence disambung dari hitungan SEJATI dump supaya lengan prevalensi gate C
    # tetap bisa diuji; ia bukan fitur kepala, jadi tidak masuk model penuh.
    Phi_h = np.column_stack([Phi_h, np.log(np.maximum(cnt_full, 1))])
    names_h = list(names_h) + ['log_prevalence']
    print('Phi_head', Phi_h.shape, '| fitur:', names_h)
    print('eksak secara konstruksi (stabilitas 1,0 adalah tautologi, bukan mutu):',
          list(EXACT_BY_CONSTRUCTION))

    # w_cos_knn_1 DICADANGKAN sebagai baseline jarak, seperti prof_knn_1 di keluarga
    # ruang-output, supaya gate C bukan uji submodel bersarang.
    HEAD_DISTANCE = 'w_cos_knn_1'
    feats_h = [n for n in names_h if n not in (HEAD_DISTANCE, 'log_prevalence')]
    print('baseline jarak dicadangkan:', HEAD_DISTANCE, '| model penuh p =', len(feats_h))

    boot_h = predictability_class_bootstrap(Phi_h, d_prim, names_h,
                                           feature_subset=feats_h,
                                           n_boot=N_BOOT_CLASS, n_splits=10, seed=SEED,
                                           reliability=r_delta)
    print()
    print('GATE B (phi kepala, bootstrap tingkat-KELAS)')
    print('  R2', format(boot_h['mean'], '+.4f'),
          'CI [' + format(boot_h['ci_low'], '+.4f') + ',',
          format(boot_h['ci_high'], '+.4f') + ']',
          '| n_kelas', boot_h['n_classes'])
    print('  ter-normalisasi oleh r_delta saja:',
          format(boot_h.get('normalized_mean', float('nan')), '+.4f'))
    gate_B_head = bool(boot_h.get('gate_B_pass_class_unit'))
    print('  gate B:', 'LULUS' if gate_B_head else 'GAGAL')

    spl_h = predictability(Phi_h, d_prim, names_h, feature_subset=feats_h,
                          distance_col=(HEAD_DISTANCE,), n_splits=N_SPLITS_BC, seed=SEED)
    print('  ablasi tersedia:', sorted(spl_h['gate_C_detail']))
    print('  baseline jarak:', spl_h['distance_col_used'],
          '| bersarang:', spl_h['distance_baseline_is_nested_in_full'])

    perm_h, pv_h, lb_h = {}, [], []
    for abl in ('distance_only', 'log_prevalence_only'):
        det = spl_h['gate_C_detail'].get(abl)
        if det is None:
            continue
        if det.get('vacuous'):
            print('  ' + abl + ': HAMPA, dikeluarkan dari verdict')
            continue
        r = class_permutation_p(Phi_h, d_prim, names_h, feature_subset=feats_h,
                                ablation=abl, distance_col=(HEAD_DISTANCE,),
                                n_perm=N_PERM_CLASS, n_splits=20, seed=SEED)
        perm_h[abl] = r
        pv_h.append(r['p_value'])
        lb_h.append(abl)
        print('  ' + abl.ljust(22), 'obs', format(r['observed'], '+.4f'),
              'null', format(r['null_mean'], '+.4f'),
              'p=' + format(r['p_value'], '.4f'))
    holm_h = holm_bonferroni(pv_h, alpha=0.05) if pv_h else None
    gate_C_head = bool(holm_h and all(h['reject'] for h in holm_h))
    if holm_h:
        for k, h in zip(lb_h, holm_h):
            print('    ' + k.ljust(22), 'p=' + format(h['p_value'], '.4f'),
                  'ambang=' + format(h['threshold'], '.4f'),
                  'tolak=' + str(h['reject']))
    print('  gate C (phi kepala):', 'LULUS' if gate_C_head else 'GAGAL')

    head = {'W_shape': list(W_head.shape), 'features': names_h,
            'distance_baseline': HEAD_DISTANCE,
            'gate_B': boot_h, 'gate_B_pass': gate_B_head,
            'gate_C_permutation': perm_h, 'gate_C_holm': holm_h,
            'gate_C_pass': gate_C_head,
            'source': 'torchvision ResNet50 IMAGENET1K_V2 fc.weight',
            'independent_of_scores': True}
except Exception as e:
    head = {'error': type(e).__name__ + ': ' + str(e)}
    print('gagal:', head['error'])
    print('Bukan pemblokir gerbang primer -- dilaporkan sebagai item terbuka.')


### Bagaimana membaca perbandingan dua keluarga phi

| phi | sumber | eksogen terhadap delta_y? |
|---|---|---|
| ruang-output | matriks skor, model SAMA | **tidak** — inilah batasannya |
| bobot kepala `w_y` | ResNet-50 torchvision, model LAIN | **ya** |

Kalau `w_y` juga lolos, klaim geometri berdiri jauh lebih kuat: prediktabilitasnya tidak bisa
lagi dijelaskan sebagai estimasi distribusi skor yang sama. Kalau `w_y` gagal sementara
ruang-output lolos, itu bukti bahwa hasil ruang-output **memang** artefak sirkularitas — dan itu
temuan yang sama pentingnya, yang harus dilaporkan apa adanya.


## 13. SEKUNDER — reproduksi Clustered CP pada skor yang SAMA

Pemeriksaan kesetiaan setup. Kalau reproduksi kita cocok dengan angka terbit Ding et al.,
sitiran jadi berlandas; kalau tidak, ketidakcocokan itu sendiri yang dilaporkan.


In [ ]:
clustered = None
if RUN_CLUSTERED_CP:
    try:
        import sys
        if not os.path.isdir('/content/ccc'):
            subprocess.run(['git','clone','--depth','1',
                            'https://github.com/tiffanyding/class-conditional-conformal.git',
                            '/content/ccc'], check=True)
        sys.path.insert(0, '/content/ccc')
        # JANGAN menebak nama fungsi. Percobaan pertama mengimpor
        # `clustered_conformal` dan gagal — nama itu tebakan. Yang benar: DAFTAR
        # apa yang benar-benar ada, lalu wiring-nya ditulis dari fakta itu.
        # Pola yang sama seperti pemanggilan gdown yang juga kutebak lalu salah.
        import importlib, inspect
        found_api = {}
        for modname in ('utils.clustering_utils', 'utils.conformal_utils'):
            try:
                m = importlib.import_module(modname)
            except Exception as e:
                found_api[modname] = f'gagal impor: {type(e).__name__}: {e}'
                continue
            fns = []
            for nm, obj in vars(m).items():
                if nm.startswith('_') or not callable(obj):
                    continue
                if getattr(obj, '__module__', None) != modname:
                    continue   # buang yang cuma di-import ke modul itu
                try:
                    sig = str(inspect.signature(obj))
                except (ValueError, TypeError):
                    sig = '(?)'
                fns.append(nm + sig)
            found_api[modname] = sorted(fns)
        for modname, api in found_api.items():
            print(f'  {modname}:')
            if isinstance(api, str):
                print('    ', api)
            else:
                for f in api:
                    print('    ', f[:150])
        clustered = {'imported': True, 'api': found_api,
                     'note': 'API didaftar, belum dipanggil — wiring ditulis dari daftar ini'}
    except Exception as e:
        clustered = {'imported': False, 'error': f'{type(e).__name__}: {e}'}
        print('gagal impor:', clustered['error'])
        print('Bukan pemblokir gerbang — dilaporkan sebagai item terbuka.')
else:
    print('dilewati')


## 14. VERDICT dan laporan


In [ ]:
import time
from pcc.utils.io import write_report

def clean(o):
    if isinstance(o, dict): return {str(k): clean(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [clean(v) for v in o]
    if isinstance(o, (np.floating, np.integer)): return o.item()
    if isinstance(o, np.ndarray): return clean(o.tolist())
    return o

print('=== UJI PRIMER (prereg_imagenet_gate.md) ===')
print('  gate A :', 'LULUS' if gate_A_pass else 'GAGAL')
print('  gate B :', 'LULUS' if gate_B_pass else 'GAGAL', '(bootstrap tingkat-kelas)')
print('  gate C :', 'LULUS' if gate_C_pass else 'GAGAL', '(permutasi kelas + Holm)')
print('  Sec 6.4:', 'LULUS' if sec64['pass'] else 'TIDAK POSITIF', '(sekunder)')
print()
if gate_B_pass and gate_C_pass:
    verdict = 'LULUS'
    print('KONSEKUENSI (sudah ditetapkan): kegagalan Pl@ntNet TERKONFIRMASI sebagai daya uji.')
    print('Phase 1 dicatat lulus; lanjut ke phi embedding lalu Phase 2.')
elif gate_B_pass:
    verdict = 'GATE C GAGAL'
    print('KONSEKUENSI: delta_y terprediksi tetapi TIDAK melampaui prediktor trivial.')
    print('Sec 6.5C terpicu — pertimbangkan menghentikan klaim kebaruan.')
else:
    verdict = 'GATE B GAGAL'
    print('KONSEKUENSI: delta_y tidak terprediksi bahkan pada jumlah kelas ini.')
    print('Batasan sejati; tulis hasil negatif.')
print()
print('INGAT: phi di sini ruang OUTPUT, bukan embedding. Hasil ini TIDAK otomatis')
print('berpindah ke deskriptor embedding.')

CAVEATS = [
    'phi(y) = geometri ruang OUTPUT dari matriks skor, BUKAN phi embedding Pl@ntNet.',
    'Penyimpangan dari urutan pre-registered: dataset ini dijalankan meski Phase 1 Pl@ntNet',
    '  tidak lulus. Alasan: kegagalannya terdiagnosis sebagai daya uji dan tidak bisa diangkat',
    '  di Pl@ntNet (38 kelas/stratum). Dicatat di prereg_imagenet_gate.md.',
    ('Ablasi prevalensi HAMPA (dataset berimbang): gate C diuji HANYA lawan '
     'baseline jarak — uji yang lebih sempit.'
     if spl['prevalence_ablation_degenerate'] else
     'Ablasi prevalensi DAPAT diuji di dump ini, jadi gate C diuji lawan KEDUA '
     'baseline: jarak dan prevalensi.'),
    ('Predictability didominasi RINGKASAN SKOR langsung, bukan geometri kelas: '
     'baseline jarak sendirian menjelaskan ~' +
     format(boot['mean'] - perm_res['distance_only']['observed'], '.3f') +
     ' dari R2 ' + format(boot['mean'], '.3f') + '. phi ruang-output DAN delta_y '
     'keduanya diturunkan dari matriks skor model yang SAMA (sampel terpisah), jadi '
     'gate B di sini lebih dekat ke estimasi distribusional daripada ekstrapolasi '
     'geometrik. Klaim geometri EMBEDDING tidak tertegakkan oleh run ini.'),
    'RUN GERBANG, bukan run paper. Dataset ini jadi dataset paper hanya jika lulus, dan angka',
    '  paper harus dari porsi evaluasi yang tidak tersentuh keputusan gerbang.',
]
for c in CAVEATS: print('CAVEAT:', c)

# write_report(reports_dir, name, *, ...) — reports_dir POSISIONAL pertama, dan
# parameter waktunya `started_at`, bukan `runtime_seconds`. Notebook 03/04 sudah
# memanggilnya benar; aku menulis ulang dari ingatan alih-alih menyalin yang terbukti.
rep_name = '05_imagenet_gate_' + best['src'] + '_' + best['ds']
path = write_report('pcc/reports', rep_name,
    hypothesis='delta_y is predictable from CLASS-LEVEL OUTPUT-SPACE geometry beyond trivial '
               'predictors, at a class count adequate to detect a weak effect',
    pass_criteria='PRIMARY per reports/prereg_imagenet_gate.md: stable feature set, n_cal=25, '
                  'alpha=0.10, all classes unstratified. Gate B = class-level bootstrap CI low '
                  '> 0 (n_boot=400). Gate C = class-level permutation null (n_perm=1000) vs '
                  'log_prevalence_only and distance_only_prereg, Holm-Bonferroni corrected. '
                  'Split-level CIs and the CI-excludes-0 criterion are NOT used: the unit of '
                  'exchangeability is the class, and 0 is not the null of the paired difference.',
    config=dict(dataset=best['ds'], source=best['src'],
                dump=os.path.abspath(best['scores']),
                n_classes=K, alpha_primary=ALPHA_PRIMARY,
                n_cal_primary=N_CAL_PRIMARY, n_boot_class=N_BOOT_CLASS,
                n_perm_class=N_PERM_CLASS, alphas_secondary=list(ALPHAS_SECONDARY),
                n_cal_secondary=N_CAL_SECONDARY, stable_threshold=STABLE_THRESHOLD,
                frac_desc=FRAC_DESC, frac_cal=FRAC_CAL,
                descriptor_family='output_space', feature_sets=clean(FEATURE_SETS),
                scores_source=f"{best['src'].upper()} released dump (no forward pass)",
                dump_survey=[{k: v for k, v in r.items()
                              if k not in ('scores','labels')} for r in cands],
                amendments=['#amendment-10']),
    seed=SEED,
    results=clean(dict(premise_ok=PREMISE_OK, dump_accuracy=acc,
                       samples_per_class=dict(min=int(cnt.min()), median=float(np.median(cnt)),
                                              max=int(cnt.max())),
                       split_sizes=dict(desc=len(i_desc), cal=len(i_cal), eval=len(i_eval)),
                       stability=stab, stable_set=stable_names,
                       gate_A=dict(reliability=r_delta, ci=[ciA['ci_low'], ciA['ci_high']],
                                   pass_=gate_A_pass),
                       ceilings=dict(r_delta=r_delta, r_phi=r_phi, joint=r_delta*r_phi),
                       gate_B_class_bootstrap=boot, gate_B_split_secondary=sf,
                       gate_C_permutation=perm_res, gate_C_holm=holm,
                       gate_C_ablations_tested=ABLATIONS_PRIMARY,
                       prevalence_ablation_degenerate=bool(spl['prevalence_ablation_degenerate']),
                       distance_baseline=DISTANCE_BASELINE,
                       gate_B_pass=gate_B_pass, gate_C_pass=gate_C_pass,
                       multi_alpha=multi, secondary=sec, sec_6_4=sec64,
                       clustered_cp=clustered, head_weight_phi=head,
                       caveats=CAVEATS)),
    conclusion=verdict)
print()
print('laporan:', path)
print('VERDICT:', verdict)
